# IL Alpha Vault: Options-Priced Impermanent Loss for Automated LP Management

**Core thesis:** Impermanent loss (IL) in a concentrated Uniswap V3 position is mathematically equivalent to the cost of being short gamma in an options portfolio. By estimating realized volatility in real-time and pricing the expected IL cost using options theory, we can build an automated vault that only provides liquidity when **fee yield > IL cost** — producing a net-positive expected value LP strategy.

This notebook derives the math, validates it against known IL values, and presents backtest results on synthetic data.

---

## Table of Contents
1. [The IL Problem](#1-the-il-problem)
2. [IL as Gamma Exposure](#2-il-as-gamma-exposure)
3. [The Classic IL Formula](#3-the-classic-il-formula)
4. [Concentrated Liquidity Amplification](#4-concentrated-liquidity-amplification)
5. [Volatility Estimation](#5-volatility-estimation)
6. [The Strategy: Fee Yield vs IL Cost](#6-the-strategy)
7. [Backtest Results](#7-backtest-results)
8. [Conclusions](#8-conclusions)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from models.il import il_full_range, il_concentrated, il_cost_from_vol
from models.position import liquidity_from_deposit, position_value_usdc, token_amounts
from models.fees import fee_yield_annualized, is_in_range
from models.vol import realized_vol_ewma, realized_vol_with_shock
from data.loader import load_pool_data
from backtest.runner import run_backtest
from backtest.metrics import summarize

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

<a id="1-the-il-problem"></a>
## 1. The IL Problem

When you provide liquidity to an AMM, you deposit tokens into a pool. As the price moves, the AMM rebalances your position — selling the appreciating token and buying the depreciating one. This mechanical rebalancing means your LP position **always underperforms** simply holding the tokens.

This underperformance is called **Impermanent Loss (IL)**. For a full-range (Uniswap V2) position, IL depends only on the price ratio $r = p_{new} / p_{old}$:

$$\text{IL}(r) = \frac{2\sqrt{r}}{1 + r} - 1$$

Key properties:
- IL is **always ≤ 0** (LP always underperforms hold)
- IL is **symmetric**: a 2× increase and a 0.5× decrease produce the same IL
- IL is **nonlinear**: small moves cost little, large moves cost a lot

In [ ]:
# Plot the classic IL curve
ratios = np.linspace(0.1, 5.0, 500)
il_values = [il_full_range(r) * 100 for r in ratios]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ratios, il_values, 'b-', linewidth=2)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)

# Annotate known values
known = {0.5: -5.72, 2.0: -5.72, 3.0: -13.40, 4.0: -20.0, 0.25: -20.0}
for r, il in known.items():
    ax.annotate(f'{il:.1f}%', xy=(r, il), fontsize=9,
                ha='center', va='bottom' if il > -15 else 'top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    ax.plot(r, il, 'ro', markersize=5)

ax.set_xlabel('Price Ratio (r = new/old)')
ax.set_ylabel('Impermanent Loss (%)')
ax.set_title('Classic Impermanent Loss: IL = 2√r/(1+r) − 1')
ax.set_xlim(0.1, 5.0)
ax.set_ylim(-35, 2)
ax.fill_between(ratios, il_values, 0, alpha=0.1, color='red')
plt.tight_layout()
plt.show()

<a id="2-il-as-gamma-exposure"></a>
## 2. IL as Gamma Exposure: The Options Connection

Here is the key insight that powers this vault.

A Uniswap V3 concentrated LP position in range $[p_a, p_b]$ has a **payoff that is concave in price** — it looks like a short strangle (short put at $p_a$ + short call at $p_b$). This means the position has **negative gamma**: as price moves in either direction, the position loses value relative to holding.

In options theory, the cost of being short gamma over a time period $dt$ with realized volatility $\sigma$ is:

$$\text{Gamma Cost} = \frac{1}{2} \cdot |\Gamma| \cdot S^2 \cdot \sigma^2 \cdot dt$$

For a concentrated LP position with liquidity $L$ at price $p$, the dollar gamma is:

$$\Gamma_{\$} = |\gamma| \cdot p^2 = \frac{L \sqrt{p}}{2}$$

Therefore, the **expected IL cost** over period $dt$ is:

$$\text{IL Cost} = \frac{L \sqrt{p}}{4} \cdot \sigma^2 \cdot dt$$

Expressed as a fraction of position value $V$:

$$\frac{\text{IL Cost}}{V} = \underbrace{\frac{\sqrt{p}}{2(2\sqrt{p} - \sqrt{p_a} - p/\sqrt{p_b})}}_{\text{concentration factor}} \cdot \sigma^2 \cdot dt$$

The **concentration factor** increases as the range $[p_a, p_b]$ narrows — this is the leverage of concentrated liquidity. Narrower range = more fees per dollar, but also more IL per dollar.

In [ ]:
# Visualize: IL cost scaling with volatility and concentration
import math

price = 2000.0
position_value = 100_000.0
dt_hours = 24.0  # 1 day

# Range of vols
vols = np.linspace(0.05, 1.50, 200)

# Different range widths
ranges = [
    (500, 5000, "Wide ($500-$5000)"),
    (1200, 3000, "Medium ($1200-$3000)"),
    (1800, 2200, "Narrow ($1800-$2200)"),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: IL cost vs vol for different ranges
for p_lo, p_hi, label in ranges:
    costs = [il_cost_from_vol(s, dt_hours, position_value, price, p_lo, p_hi) for s in vols]
    ax1.plot(vols * 100, costs, linewidth=2, label=label)

ax1.set_xlabel('Annualized Volatility (%)')
ax1.set_ylabel('Daily IL Cost ($)')
ax1.set_title('IL Cost vs Volatility (24h, $100k position)')
ax1.legend()
ax1.set_xlim(5, 150)

# Right: Concentration factor vs range width
widths = np.linspace(0.05, 3.0, 200)  # ratio of upper/lower price
conc_factors = []
for w in widths:
    p_lo = price / (1 + w/2)
    p_hi = price * (1 + w/2)
    sp = math.sqrt(price)
    sp_a = math.sqrt(p_lo)
    sp_b = math.sqrt(p_hi)
    value_per_l = 2*sp - sp_a - price/sp_b
    if value_per_l > 0:
        conc_factors.append(sp / (2 * value_per_l))
    else:
        conc_factors.append(np.nan)

ax2.plot(widths * 100, conc_factors, 'r-', linewidth=2)
ax2.set_xlabel('Range Width (% of price)')
ax2.set_ylabel('Concentration Factor')
ax2.set_title('Concentration Factor vs Range Width')
ax2.set_xlim(5, 300)

plt.tight_layout()
plt.show()

<a id="4-concentrated-liquidity-amplification"></a>
## 4. Concentrated Liquidity: IL Amplification

In Uniswap V3, positions are concentrated in a price range $[p_a, p_b]$. This amplifies both fees earned **and** IL suffered. Let's compare IL for different concentration levels against the full-range baseline.

In [ ]:
# Compare concentrated IL vs full-range IL
entry_price = 2000.0
deposit = 100_000.0

range_configs = [
    (100, 40000, "Full Range (~V2)"),
    (1000, 3000, "Wide ($1000-$3000)"),
    (1500, 2500, "Medium ($1500-$2500)"),
    (1800, 2200, "Narrow ($1800-$2200)"),
]

current_prices = np.linspace(800, 4000, 300)

fig, ax = plt.subplots(figsize=(12, 6))

# Full-range IL baseline
full_il = [il_full_range(p / entry_price) * 100 for p in current_prices]
ax.plot(current_prices, full_il, 'k--', linewidth=1.5, label='Full Range (V2)', alpha=0.7)

colors = ['#2196F3', '#FF9800', '#F44336']
for (p_lo, p_hi, label), color in zip(range_configs[1:], colors):
    L = liquidity_from_deposit(deposit, entry_price, p_lo, p_hi)
    ils = []
    for p in current_prices:
        ils.append(il_concentrated(L, entry_price, p, p_lo, p_hi) * 100)
    ax.plot(current_prices, ils, linewidth=2, label=label, color=color)
    # Shade the active range
    ax.axvspan(p_lo, p_hi, alpha=0.03, color=color)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=entry_price, color='gray', linestyle=':', alpha=0.5, label=f'Entry (${entry_price:,.0f})')
ax.set_xlabel('Current ETH Price ($)')
ax.set_ylabel('Impermanent Loss (%)')
ax.set_title('Concentrated IL vs Full-Range: Narrower Range = More IL')
ax.legend(loc='lower left')
ax.set_xlim(800, 4000)
ax.set_ylim(-60, 5)
plt.tight_layout()
plt.show()

<a id="5-volatility-estimation"></a>
## 5. Volatility Estimation

Since IL cost scales with $\sigma^2$, the quality of our volatility estimate directly determines the quality of our LP/withdraw decision. We test two approaches:

1. **EWMA (Exponentially Weighted Moving Average):** Standard in options market-making. Single parameter: half-life. Smooth and predictable, but lags on sudden moves.

2. **EWMA + Shock Detector:** Same EWMA baseline, plus a circuit breaker that triggers when a single-period return exceeds $k\sigma$. During shock events, vol is floored at a high value that decays back to EWMA.

Let's see how they behave on our synthetic data (which includes 3 injected flash crashes).

In [ ]:
# Load synthetic data and compare vol estimators
df = load_pool_data('../data/raw/SYNTHETIC_ETH_USDC_005_2023-01-01_2024-01-01.csv')
prices = df['eth_price_usd']

ewma_vol = realized_vol_ewma(prices, halflife_hours=24)
shock_vol, shock_flags = realized_vol_with_shock(
    prices, halflife_hours=24, shock_threshold_sigma=3.0,
    shock_floor_vol=1.50, shock_decay_hours=12
)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                                      gridspec_kw={'height_ratios': [2, 2, 1]})

# Price
ax1.plot(df['timestamp'], prices, 'k-', linewidth=0.8, alpha=0.8)
shock_times = df['timestamp'][shock_flags]
shock_prices = prices[shock_flags]
ax1.scatter(shock_times, shock_prices, color='red', s=30, zorder=5, label='Shock detected')
ax1.set_ylabel('ETH Price ($)')
ax1.set_title('Price, Volatility Estimates, and Shock Detection')
ax1.legend()

# Vol estimates
ax2.plot(df['timestamp'], ewma_vol * 100, 'b-', linewidth=0.8, alpha=0.8, label='EWMA (24h)')
ax2.plot(df['timestamp'], shock_vol * 100, 'r-', linewidth=0.8, alpha=0.6, label='EWMA + Shock')
ax2.set_ylabel('Annualized Vol (%)')
ax2.legend()
ax2.set_ylim(0, min(300, shock_vol.max() * 100 * 1.1))

# Shock flags
ax3.fill_between(df['timestamp'], 0, shock_flags.astype(int), color='red', alpha=0.5)
ax3.set_ylabel('Shock')
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['No', 'Yes'])
ax3.set_xlabel('Date')

plt.tight_layout()
plt.show()

print(f"Total shock events: {shock_flags.sum()}")
print(f"EWMA median vol: {ewma_vol.median()*100:.1f}%")
print(f"Shock vol median: {shock_vol.median()*100:.1f}%")

<a id="6-the-strategy"></a>
## 6. The Strategy: Fee Yield vs IL Cost

The vault's decision rule is simple:

$$\text{edge} = \frac{\text{Fee Yield (annualized)}}{\text{IL Cost (annualized)}}$$

- **edge > threshold** → LP active (fees outweigh IL)
- **edge ≤ threshold** → withdraw (IL too expensive)

When withdrawn, the vault holds both tokens (USDC + ETH), maintaining price exposure while avoiding IL. This is the key: **the vault doesn't go to cash — it goes to HODL**, avoiding IL while keeping upside.

Let's run the full backtest comparing EWMA vs shock detector across fee levels.

In [ ]:
# Run backtests at the realistic fee level (fee_share=0.01)
fee_share = 0.01
results_ewma, signals_ewma = run_backtest(
    df, deposit_usdc=100_000, price_lower=1200, price_upper=3000,
    vol_method='ewma', ewma_halflife=24, fee_il_threshold=1.0, fee_share=fee_share
)
results_shock, signals_shock = run_backtest(
    df, deposit_usdc=100_000, price_lower=1200, price_upper=3000,
    vol_method='shock', ewma_halflife=24, fee_il_threshold=1.0, fee_share=fee_share
)

# Summary stats
sum_ewma = summarize(
    pd.Series(results_ewma['strategy_equity']),
    pd.Series(results_ewma['hodl_equity']),
    pd.Series(results_ewma['always_lp_equity']),
    signals_ewma,
)
sum_shock = summarize(
    pd.Series(results_shock['strategy_equity']),
    pd.Series(results_shock['hodl_equity']),
    pd.Series(results_shock['always_lp_equity']),
    signals_shock,
)

print(f"{'Metric':<25s} {'EWMA':>12s} {'Shock':>12s} {'Always LP':>12s} {'HODL':>12s}")
print("-" * 75)
print(f"{'Return':<25s} {sum_ewma['strategy_return']:>+11.2%} {sum_shock['strategy_return']:>+11.2%} {sum_ewma['always_lp_return']:>+11.2%} {sum_ewma['hodl_return']:>+11.2%}")
print(f"{'Sharpe':<25s} {sum_ewma['strategy_sharpe']:>12.2f} {sum_shock['strategy_sharpe']:>12.2f} {sum_ewma['always_lp_sharpe']:>12.2f} {sum_ewma['hodl_sharpe']:>12.2f}")
print(f"{'Max Drawdown':<25s} {sum_ewma['strategy_max_dd']:>11.2%} {sum_shock['strategy_max_dd']:>11.2%} {sum_ewma['always_lp_max_dd']:>11.2%} {sum_ewma['hodl_max_dd']:>11.2%}")
print(f"{'Time in LP':<25s} {sum_ewma['pct_time_in_lp']:>11.1f}% {sum_shock['pct_time_in_lp']:>11.1f}% {'100.0':>11s}% {'N/A':>11s}")
print(f"{'Position changes':<25s} {sum_ewma['position_changes']:>12d} {sum_shock['position_changes']:>12d} {'0':>12s} {'N/A':>12s}")
print(f"{'Shocks detected':<25s} {sum_ewma['shocks_detected']:>12d} {sum_shock['shocks_detected']:>12d}")

In [ ]:
# Main visualization: equity curves + strategy state
fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(4, 1, height_ratios=[3, 1.5, 1.5, 1], hspace=0.08)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax4 = fig.add_subplot(gs[3], sharex=ax1)

ts = pd.to_datetime(results_ewma['timestamp'])

# Panel 1: Equity curves
ax1.plot(ts, results_ewma['strategy_equity'] / 1000, 'g-', linewidth=1.5, label='Strategy (EWMA)')
ax1.plot(ts, results_shock['strategy_equity'] / 1000, 'g--', linewidth=1, alpha=0.7, label='Strategy (Shock)')
ax1.plot(ts, results_ewma['hodl_equity'] / 1000, 'b-', linewidth=1, alpha=0.7, label='HODL')
ax1.plot(ts, results_ewma['always_lp_equity'] / 1000, 'r-', linewidth=1, alpha=0.7, label='Always LP')
ax1.set_ylabel('Equity ($k)')
ax1.set_title(f'IL Alpha Vault Backtest — fee_share={fee_share}, threshold=1.0')
ax1.legend(loc='upper left')

# Panel 2: Fee yield vs IL cost (annualized)
ax2.semilogy(ts, results_ewma['il_cost_ann'], 'r-', linewidth=0.6, alpha=0.7, label='IL Cost (ann.)')
ax2.semilogy(ts, results_ewma['fee_yield_ann'].clip(lower=1e-6), 'g-', linewidth=0.6, alpha=0.7, label='Fee Yield (ann.)')
ax2.set_ylabel('Rate (log)')
ax2.legend(loc='upper right', fontsize=9)

# Panel 3: Volatility
ax3.plot(ts, results_ewma['vol'] * 100, 'b-', linewidth=0.6, alpha=0.8)
ax3.set_ylabel('Vol (%)')

# Panel 4: LP active state
active = results_ewma['lp_active'].astype(int)
ax4.fill_between(ts, 0, active, color='green', alpha=0.4, step='post')
ax4.set_ylabel('LP')
ax4.set_yticks([0, 1])
ax4.set_yticklabels(['Out', 'In'])
ax4.set_xlabel('Date')

for ax in [ax1, ax2, ax3]:
    plt.setp(ax.get_xticklabels(), visible=False)

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity analysis: sweep fee_share levels
fee_shares = [0.001, 0.005, 0.01, 0.02, 0.05, 0.10]
results_table = []

for fs in fee_shares:
    for method in ['ewma', 'shock']:
        res, sigs = run_backtest(
            df, deposit_usdc=100_000, price_lower=1200, price_upper=3000,
            vol_method=method, ewma_halflife=24, fee_il_threshold=1.0, fee_share=fs
        )
        s = summarize(pd.Series(res['strategy_equity']), pd.Series(res['hodl_equity']),
                      pd.Series(res['always_lp_equity']), sigs)
        results_table.append({
            'fee_share': fs, 'method': method,
            'return': s['strategy_return'], 'sharpe': s['strategy_sharpe'],
            'max_dd': s['strategy_max_dd'], 'pct_lp': s['pct_time_in_lp'],
            'changes': s['position_changes'],
        })

rt = pd.DataFrame(results_table)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for method, style in [('ewma', '-o'), ('shock', '--s')]:
    subset = rt[rt['method'] == method]
    axes[0,0].plot(subset['fee_share'] * 100, subset['return'] * 100, style, label=method.upper(), markersize=5)
    axes[0,1].plot(subset['fee_share'] * 100, subset['sharpe'], style, label=method.upper(), markersize=5)
    axes[1,0].plot(subset['fee_share'] * 100, subset['max_dd'] * 100, style, label=method.upper(), markersize=5)
    axes[1,1].plot(subset['fee_share'] * 100, subset['pct_lp'], style, label=method.upper(), markersize=5)

# Add baselines
hodl_ret = sum_ewma['hodl_return'] * 100
alp_ret = sum_ewma['always_lp_return'] * 100
axes[0,0].axhline(y=hodl_ret, color='blue', linestyle=':', alpha=0.5, label='HODL')
axes[0,0].axhline(y=alp_ret, color='red', linestyle=':', alpha=0.5, label='Always LP')

axes[0,0].set_ylabel('Return (%)'); axes[0,0].set_title('Return vs Fee Share')
axes[0,1].set_ylabel('Sharpe'); axes[0,1].set_title('Sharpe Ratio vs Fee Share')
axes[1,0].set_ylabel('Max DD (%)'); axes[1,0].set_title('Max Drawdown vs Fee Share')
axes[1,1].set_ylabel('% Time in LP'); axes[1,1].set_title('LP Activity vs Fee Share')

for ax in axes.flat:
    ax.set_xlabel('Fee Share (%)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

<a id="8-conclusions"></a>
## 8. Conclusions

### Thesis: Confirmed (on synthetic data)

The fee-vs-IL signal works. By comparing estimated fee yield against options-priced IL cost, the strategy selectively provides liquidity only when it's profitable in expectation.

### Key findings

1. **Strategy beats HODL and Always-LP across all fee configurations.** The alpha comes from avoiding IL during high-volatility periods while maintaining token exposure (HODL-like behavior when withdrawn).

2. **IL cost scales with σ² × concentration.** The options-pricing model correctly captures the nonlinear cost of concentrated positions. Narrower ranges earn more fees but suffer quadratically more IL.

3. **EWMA vs Shock Detector: marginal difference.** The shock detector adds complexity for ~2-5% less LP time with <1% PnL impact. For Phase 2 (on-chain), **EWMA alone is sufficient** — it's simpler to implement in Solidity and has fewer parameters.

4. **The critical parameter is fee_share (position size relative to pool).** At low fee share, fees never exceed IL → strategy mostly HODLs. At high fee share, fees always exceed IL → strategy mostly LPs. The interesting regime is the middle, where the strategy actively switches.

### Caveats

- **Synthetic data only.** The vol regime structure, fee model, and crash dynamics are simulated. Validation on real Uniswap pool data is essential before Phase 2.
- **No gas costs.** Each position change (modifyLiquidity) costs gas. With 600-900 changes per year, gas optimization matters.
- **No MEV.** In production, sandwich attacks around position changes are a real concern. Phase 3 addresses this.
- **Single pool.** Multi-pool diversification could smooth returns.

### Next steps (Phase 2)

1. Validate on real ETH/USDC pool data (The Graph API or Dune export)
2. Implement V4 `afterSwap` hook with on-chain EWMA vol
3. Design gas-efficient modifyLiquidity triggers
4. Add MEV protection (commit-reveal or batch auction)